In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ipywidgets import interact, interactive, FloatSlider, VBox, Output
from IPython.display import display

In [ ]:
from rail.estimation.algos import bpz_lite
from rail.utils import catalog_utils

model_file = 'pz_challenge_taskset_1_cardinal_pz_model_10yr.pkl'

In [ ]:
catalog_utils.load_yaml("catalogs.yaml")
catalog_utils.apply("cardinal_roman_rubin")

estimator = bpz_lite.BPZliteEstimator.make_stage(
    name=f"estimate",
    model=model_file,
    output_mode="return",
    spectra_file="COSMOS_seds.list",
)
estimator.open_model(**estimator.config)
estimator.validate()
estimator._initialize_run()

In [ ]:
mag_u = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_u:', continuous_update=True)
mag_g = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_g:', continuous_update=True)
mag_r = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_r:', continuous_update=True)
mag_i = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_i:', continuous_update=True)
mag_z = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_z:', continuous_update=True)
mag_y = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_y:', continuous_update=True)
mag_Y = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_Y:', continuous_update=True)
mag_J = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_J:', continuous_update=True)
mag_H = FloatSlider(min=20, max=25, step=0.05, value=22.0, description='mag_H:', continuous_update=True)
err_level = FloatSlider(min=0.01, max=0.2, step=0.01, value=0.1, description='err level:', continuous_update=True)


In [ ]:
xvals = np.linspace(0., 1.5, 151)
fig = go.FigureWidget()
fig.add_trace(go.Scatter(x=xvals, y=np.zeros_like(xvals), mode='lines'))
fig.update_layout(title='PDF', xaxis_title='X', yaxis_title='Y')

def update_plot(mag_u, mag_g, mag_r, mag_i, mag_z, mag_y, mag_Y, mag_J, mag_H, err_level):
    dd = dict(
        mag_u_lsst=np.array([mag_u]),
        mag_g_lsst=np.array([mag_g]),
        mag_r_lsst=np.array([mag_r]),
        mag_i_lsst=np.array([mag_i]),
        mag_z_lsst=np.array([mag_z]),
        mag_y_lsst=np.array([mag_y]),
        mag_Y_roman=np.array([mag_Y]),
        mag_J_roman=np.array([mag_J]),
        mag_H_roman=np.array([mag_H]),    
        mag_u_lsst_err=np.array([err_level]),
        mag_g_lsst_err=np.array([err_level]),
        mag_r_lsst_err=np.array([err_level]),
        mag_i_lsst_err=np.array([err_level]),
        mag_z_lsst_err=np.array([err_level]),
        mag_y_lsst_err=np.array([err_level]),
        mag_Y_roman_err=np.array([err_level]),
        mag_J_roman_err=np.array([err_level]),
        mag_H_roman_err=np.array([err_level]),        
    )
    estimator.data_store.clear()
    estimator.validate()    
    estimator._initialize_run()
    estimator._output_handle = None        
    estimator._process_chunk(0, 1, dd, True)
    handle = estimator.get_handle('output')
    yvals = np.squeeze(handle.data.pdf(xvals))
    
    # Update trace data in-place instead of recreating figure
    with fig.batch_update():
        fig.data[0].y = yvals

w = interactive(
    update_plot,
    mag_u=mag_u, mag_g=mag_g, mag_r=mag_r,
    mag_i=mag_i, mag_z=mag_z, mag_y=mag_y,
    mag_Y=mag_Y, mag_J=mag_J, mag_H=mag_H,
    err_level=err_level,
)

In [ ]:
# Display the widget
display(w, fig)

In [ ]:
def plot_colors(mag_u, mag_g, mag_r, mag_i, mag_z, mag_y):
    """
    Plot colors
    """
    u_g = mag_u - mag_g
    g_r = mag_g - mag_r
    r_i = mag_r - mag_i
    i_z = mag_i - mag_z
    z_y = mag_z - mag_y
    
    # Create the plot
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('g-r v. u-g', 'r-i v. g-r', 'i-z v. r-i', 'z-y v. i-z'),
        shared_xaxes=True,
        shared_yaxes=True,
    )
    
    fig.add_trace(go.Scatter(x=[u_g], y=[g_r]), row=1, col=1)
    fig.add_trace(go.Scatter(x=[g_r], y=[r_i]), row=1, col=2)
    fig.add_trace(go.Scatter(x=[r_i], y=[i_z]), row=2, col=1)
    fig.add_trace(go.Scatter(x=[i_z], y=[z_y]), row=2, col=2)

    fig.update_xaxes(title_text="u-g", row=1, col=1)
    fig.update_xaxes(title_text="g-r", row=1, col=2)
    fig.update_xaxes(title_text="r-i", row=2, col=1)
    fig.update_xaxes(title_text="i-z", row=2, col=2)

    fig.update_yaxes(title_text="g-r", row=1, col=1)
    fig.update_yaxes(title_text="r-i", row=1, col=2)
    fig.update_yaxes(title_text="i-z", row=2, col=1)
    fig.update_yaxes(title_text="z-y", row=2, col=2)

    fig.update_xaxes(range=[-5, 5])
    fig.update_yaxes(range=[-5, 5])

    fig.update_layout(
        height=600,
        width=600,
        title='Color-color',
        xaxis_range=[-5, 5],
        yaxis_range=[-5, 5],
        showlegend=False,
        hovermode='x'
    )
    
    fig.show()

In [ ]:
w = interactive(
    plot_colors,
    mag_u=mag_u,
    mag_g=mag_g,
    mag_r=mag_r,
    mag_i=mag_i,
    mag_z=mag_z,
    mag_y=mag_y,
)

In [ ]:
# Display the widget
display(w)